# 10. Lecke - AI Ügynökök éles környezetben

Ebben a leckében az AI ügynökök **éles környezetbeli mintáit** tanulhatod meg a Microsoft Agent Framework (Python) használatával. A következőket tárgyaljuk:

- **Megfigyelhetőség** — időzítés és naplózás hozzáadása az ügynök interakciókhoz
- **Értékelés** — egy értékelő ügynök használata a válaszok minőségének pontozására
- **Költségkezelés** — stratégiák token optimalizálásra és modellválasztásra

A forgatókönyv egy **utazási ügynök**, amely segít a felhasználóknak utazások tervezésében, megfigyelés és értékelés rétegezve rá.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Termelési megfontolások

Az AI ügynökök prototípusból termelési környezetbe helyezése három pillér alapos figyelmét igényli:

1. **Megfigyelhetőség** — Láthatóság szükséges arra, mit csinál az ügynök, mennyi ideig tart, és milyen eszközöket hív meg. Követés és naplózás nélkül a termelési problémák hibakeresése szinte lehetetlen.

2. **Értékelés** — Az automatizált minőségellenőrzés biztosítja, hogy az ügynök válaszai idővel pontosak, teljesek és hasznosak maradjanak. Egy értékelő ügynök pontozhatja a válaszokat a meghatározott kritériumok alapján.

3. **Költségkezelés** — A tokenhasználat közvetlenül befolyásolja a költségeket. Olyan stratégiák, mint a prompt optimalizálás, modellválasztás és gyorsítótárazás segítenek a kiadások kordában tartásában a minőség feláldozása nélkül.


## Egy megfigyelhető ügynök létrehozása

Meghatározzuk az utazási eszközöket, és az ügynök hívását időzítéssel vesszük körül, hogy figyelemmel kísérhessük a késleltetést. Éles környezetben OpenTelemetry-vel vagy hasonló nyomkövetési háttérrendszerrel integrálnánk.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Értékelési minták

Egy gyakori termelési minta a második ügynök használata mint **értékelő**. Az értékelő a fő ügynök válaszát előre meghatározott kritériumok alapján pontozza, mint például teljesség, pontosság és hasznosság.

Ez lehetővé teszi:
- Automatizált minőségi kapuk alkalmazását, mielőtt a válaszok elérnék a felhasználókat
- Regresszió észlelését, amikor a promptok vagy modellek változnak
- Az ügynök teljesítményének folyamatos monitorozását az idő múlásával


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Költségkezelési stratégiák

A költségek ellenőrzése kritikus az AI ügynökök esetében. Íme a kulcsfontosságú stratégiák:

| Stratégia | Leírás |
|---|---|
| **Prompt optimalizálás** | Legyenek tömörek a rendszer utasítások. Távolítsuk el a felesleges kontextust az input tokenek csökkentése érdekében. |
    "| **Modell kiválasztása** | Egyszerű feladatokhoz, mint osztályozás vagy kinyerés, használjunk kisebb, olcsóbb modelleket (pl. GPT-4.1-mini), és tartsuk fenn a nagyobb modelleket összetettebb gondolkodáshoz. |\n",
| **Gyorsítótárazás** | Tároljuk az eszköz eredményeket és gyakori lekérdezéseket, hogy elkerüljük az ismételt API hívásokat. |
| **Token költségvetés** | Állítsunk be `max_tokens` korlátokat a váratlanul hosszú válaszok megelőzésére. |
| **Csomagolás** | Lehetőség szerint csoportosítsuk a felhasználói lekérdezéseket egyetlen API hívásba. |

A gyakorlatban egy többszintű megközelítés működik jól: az egyszerű kéréseket gyors, olcsó modellhez irányítjuk, és csak az összetett lekérdezéseket továbbítjuk képzettebb modellhez.


## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Adj megfigyelhetőséget** az ügynökök interakcióihoz időzítés és naplózás segítségével, megalapozva a nyomon követést és monitorozást.
2. **Értékeld automatikusan az ügynök válaszait** egy értékelő ügynökkel, amely a teljességet, pontosságot és hasznosságot pontozza.
3. **Kezeld a költségeket** kérésoptimalizálással, modellválasztással, gyorsítótárazással és tokenkeretekkel.

Ezek a gyártási minták segítenek biztosítani, hogy AI ügynökeid megbízhatóak, mérhetőek és költséghatékonyak legyenek nagy léptékben.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
